# RAG con GROQ sobre un documento PDF en ingles

**Actividad:** implementar un sistema de **generacion aumentada por recuperacion (RAG)** ...

## Que exige la actividad

| # | Requisito | Donde se cumple |
|---|---|---|
| 1 | Usar un **documento PDF en ingles** para vectorizar el texto | Celdas de carga del PDF y de los chunks |
| 2 | Tener cuenta creada en **GROQ** y una **API KEY** | Celda de configuracion de la API key |
| 3 | Que el modelo **llama3.1:8b** genere la respuesta de la consulta usando la **busqueda** **hibrida** | Celda de inferencia con la recuperacion hibrida |
| 4 | Realizar **3 consultas diferentes** sobre el documento con el resultado de la busqueda **hibrida** | Celdas de consultas |

## Como usar este notebook

Ejecuta las celdas **en orden** (`Runtime > Run all`, o `Ctrl+Enter` una por una).

1. La primera celda instala las dependencias.
2. La segunda celda configura la **API KEY** de GROQ.
3. Las siguientes celdas cargan el **documento PDF en ingles** y vectorizan el texto.
4. Las celdas siguientes definen las **busquedas**: densa, dispersa y hibrida.
5. La ultima celda aplica el **modelo LLM** `llama3.1:8b` a las consultas y compara resultados.

In [ ]:
!pip install --quiet groq openai rank_bm25 pypdf

## 2. Configuracion de la API KEY de GROQ

La actividad exige crear una cuenta en [GROQ](https://groq.com/) y generar una **API KEY**.

En Google Colab, abre el panel de `Tools > Secrets`, crea un secret llamado `GROQ_API_KEY` con el valor de tu clave y el notebook lo leera automaticamente.

Si prefieres no usar secrets, pega la clave directamente en la celda de codigo siguiente o define la variable de entorno `GROQ_API_KEY`.

In [ ]:
import os
import re

import numpy as np
import openai
from groq import Groq
from openai import OpenAI

try:
    from google.colab import userdata

    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

groq = Groq(api_key=GROQ_API_KEY)
embed_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY)

# Modelo de embeddings de GROQ para documentos en ingles
EMBED_MODEL = "embed-english-v3.0"

# Modelo de chat de GROQ (equivalente al `llama3.1:8b` de la actividad)
LLM_MODEL = "llama-3.1-8b-instant"

## 3. Carga del documento PDF en ingles y vectorizacion del texto

Se carga el **documento PDF en ingles** y se divide en **chunks** de texto para poder vectorizarlo y recuperar la informacion relevante.

La ruta esperada es `/content/documento.pdf`; en Google Colab se sube el archivo con el panel de `Files`.

In [ ]:
from pypdf import PdfReader

PDF_PATH = "/content/documento.pdf"  # En Colab: subir el PDF por Files > Upload

reader = PdfReader(PDF_PATH)
texto = "\n\n".join(
    f"Pagina {i + 1}:\n{page}" for i, page in enumerate(reader.pages) if (page.extract_text() or "").strip()
)


def dividir_en_chunks(texto, tamano=500, solapimiento=50):
    parrafos = re.split(r"\n\s*\n", texto.strip())
    chunks = []
    actual = ""
    for p in parrafos:
        if len(actual) + len(p) > tamano and actual.strip():
            chunks.append(actual.strip())
            actual = actual[-solapamiento:] + " " + p
        else:
            actual = (actual + " " + p).strip() if actual.strip() else p.strip()
    if actual.strip():
        chunks.append(actual.strip())
    return chunks


chunks = [{"id": i, "texto": c} for i, c in enumerate(divividir_en_chunks(texto))]
print(f"Caracteres extraidos: {len(texto)}")
print(f"Numero de chunks: {len(chunks)}")
for c in chunks:
    print(f"- Chunk {c['id']}: {len(c['texto'])} caracteres")

## 4. Embeddings y busqueda densa

Se generan los embeddings de los chunks con el modelo `embed-english-v3.0` de GROQ y se calcula la **similitud coseno** entre el embedding de la consulta y el de cada chunk para recuperar los mas relevantes.

In [ ]:
def embed_textos(textos):
    vectores = []
    for i in range(0, len(textos), 16):
        lote = textos[i : i + 16]
        respuesta = embed_client.embeddings.create(model=EMBED_MODEL, input=lote)
        vectores.extend([item.embedding for item in respuesta.data])
    return vectores


VECTORES = embed_textos([c["texto"] for c in chunks])


def similitud_coseno(a, b):
    a = np.array(a, dtype=np.float64)
    b = np.array(b, dtype=np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def buscar_densa(query, k=5):
    embedding_query = embed_textos([query])[0]
    scored = sorted(
        range(len(chunks)),
        key=lambda i: similitud_coseno(embedding_query, VECTORES[i]),
        reverse=True,
    )[:k]
    return [(i, similitud_coseno(embedding_query, VECTORES[i])) for i in scored]

## 5. Busqueda dispersa y busqueda hibrida

La **busqueda dispersa** usa **BM25** sobre los chunks para rankear por terminos. La **busqueda** hibrida combina la busqueda **densa** y la **dispersa** mediante **Reciprocal Rank Fusion (RRF)**.


In [ ]:
from rank_bm25 import BM25

tokenizado = [re.findall(r"[a-z0-9]+", c["texto"].lower()) for c in chunks]
bm25 = BM25(tokenizado)


def buscar_dispersa(query, k=5):
    tokens = re.findall(r"[a-z0-9]+", query.lower())
    scores = bm25.get_scores(tokens)
    orden = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(i, float(scores[i])) for i in orden if scores[i] > 0]


def busqueda_hibrida(query, k=5, k0=60):
    densa = buscar_densa(query, k=10)
    dispersa = buscar_dispersa(query, k=10)
    fusion = {}
    for rank, (i, _) in enumerate(densa, start=1):
        fusion[i] = fusion.get(i, 0.0) + 1.0 / (k0 + rank)
    for rank, (i, _) in enumerate(dispersa, start=1):
        fusion[i] = fusion.get(i, 0.0) + 1.0 / (k0 + rank)
    top = sorted(fusion.items(), key=lambda t: t[1], reverse=True)[:k]
    return [(i, s) for i, s in top]


def obtener_ids(query, metodo, k=5):
    if metodo == "hibrida":
        return busqueda_hibrida(query, k=k)
    if metodo == "densa":
        return buscar_densa(query, k=k)
    return buscar_dispersa(query, k=k)

## 6. Aplicacion del modelo LLM `llama3.1:8b` a las 3 consultas

Con el contexto recuperado se construye el prompt y se envia a GROQ para que el modelo genere la respuesta. A continuacion se ejecutan las **3 consultas** con las **3 busquedas** (**hibrida**, **densa** y **dispersa**) y se comparan los resultados obtenidos.

In [ ]:
def generar_respuesta(query, contexto, model=LLM_MODEL, temperature=0.2):
    texto_contexto = "\n\n".join(c["texto"] for c in contexto)
    prompt = (
        "You are a helpful assistant. Answer the question using ONLY the information contained in "
        "the provided context. If the answer is not present in the context, reply that you do not know.\n\n"
        "Context:\n" + texto_contexto + "\n\nQuestion: " + query + "\nAnswer:"
    )
    completion = groq.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=1024,
    )
    return completion.choices[0].message.content


QUERIES = [
    "What is the main topic of this document?",
    "What are the key findings or conclusions presented in the text?",
    "Summarize the most important points of the document in a short answer.",
]


def consultar(query, metodo, model=LLM_MODEL):
    ids = obtener_ids(query, metodo)
    contexto = [chunks[i] for i, _ in ids]
    respuesta = generar_respuesta(query, contexto, model=model)
    return {"query": query, "metodo": metodo, "contexto": contexto, "respuesta": respuesta, "ids": ids}


RESULTADOS = []
for query in QUERIES:
    for metodo in ("hibrida", "densa", "dispersa"):
        resultado = consultar(query, metodo)
        RESULTADOS.append(resultado)
        print(f"\n=== {metodo.upper()} | {query} ===")
        print(resultado["respuesta"])
        print(f"Chunks recuperados: {[i for i, _ in resultado['ids']]}")


print("\n=== RESULTADOS ACUMULADOS (modelo LLM: llama3.1:8b) ===")
for r in RESULTADOS:
    print(f"- {r['metodo']} | {r['query']} -> {r['respuesta']}")

## 7. Otro modelo LLM en GROQ: repetir el punto 3

Se identifica otro modelo de **chat** disponible en la plataforma **GROQ** y se repite el punto 3
con el nuevo modelo usando la **busqueda hibrida**.

In [ ]:
LLM_MODEL_SEGUNDO = "llama-3.3-70b-versatile"

RESULTADOS_SEGUNDO = []
for query in QUERIES:
    resultado = consultar(query, "hibrida", model=LLM_MODEL_SEGUNDO)
    RESULTADOS_SEGUNDO.append(resultado)
    print(f"\n=== SEGUNDO_MODELO | {query} ===")
    print(resultado["respuesta"])


print("\n=== RESULTADOS CON EL SEGUNDO MODELO (HIBRIDA) ===")
for r in RESULTADOS_SEGUNDO:
    print(f"- {r['query']} -> {r['respuesta']}")